In [ ]:
import requests
import pandas as pd

API_URL = f"https://tabular-api.data.gouv.fr/api/resources/77ed6b2f-c48f-4037-8479-50af74fa5c7a/data/"
df_election = pd.read_csv('src/election.csv', sep=';')

all_data = []
page = 1

while True:
    response = requests.get(API_URL, params={"page": page, "page_size": 200})
    result = response.json()
    all_data.extend(result["data"])
    
    if result["links"]["next"] is None:
        break
    page += 1

df = pd.DataFrame(all_data)
display(df)

In [ ]:
new_cols = list(df.columns[:2]) + list(df.iloc[2, 2:])
df.columns = new_cols
df = df.iloc[3:].reset_index(drop=True)

In [ ]:
df.columns

In [ ]:
CLASSIFICATION = {
    "ARTHAUD":       "gauche", 
    "POUTOU":        "gauche",
    "HAMON":         "gauche",
    "MÉLENCHON":     "gauche",
    "MACRON":        "centre",
    "FILLON":        "droite",
    "LE PEN":        "droite",
    "DUPONT-AIGNAN": "droite",
    "ASSELINEAU":    "droite",
    "LASSALLE":      "centre",
    "CHEMINADE":     "droite",
}

all_cols = list(df.columns)
nom_indices = [i for i, c in enumerate(all_cols) if c == "Nom"]
pct_indices = [i for i, c in enumerate(all_cols) if c == "% Voix/Exp"]

df["% gauche/Exp"] = 0.0
df["% centre/Exp"] = 0.0
df["% droite/Exp"] = 0.0

for nom_idx, pct_idx in zip(nom_indices, pct_indices):
    nom = df.iloc[0, nom_idx]
    bloc = CLASSIFICATION.get(nom, "autre")
    pct = pd.to_numeric(df.iloc[:, pct_idx], errors="coerce").fillna(0)
    
    if bloc == "gauche":
        df["% gauche/Exp"] = df["% gauche/Exp"] + pct
    elif bloc == "centre":
        df["% centre/Exp"] = df["% centre/Exp"] + pct
    elif bloc == "droite":
        df["% droite/Exp"] = df["% droite/Exp"] + pct

df_out = df[["Libellé de la commune", "% gauche/Exp", "% centre/Exp", "% droite/Exp",
    "Inscrits",
    "Abstentions",
    "% Abs/Ins",
    "Votants",
    "% Vot/Ins",
    "Blancs",
    "% Blancs/Ins",
    "% Blancs/Vot",
    "Nuls",
    "% Nuls/Ins",
    "% Nuls/Vot",
    "Exprimés",
    "% Exp/Ins",
    "% Exp/Vot",]]

display(df_out) 

In [ ]:
df_out['Année'] = 2017
display(df_out)

In [ ]:
df_out = df_out.merge(
    df_election[["Libellé de la commune", "Code_INSEE"]],
    on="Libellé de la commune",
    how="left"
)

In [ ]:
df_out["Résultat"] = df_out[["% gauche/Exp", "% centre/Exp", "% droite/Exp"]].idxmax(axis=1).str.replace("% ", "").str.replace("/Exp", "")

In [ ]:
display(df_out)

In [ ]:

df_out = df_out.dropna()
display(df_out)

In [ ]:
df_out.drop_duplicates(subset=['Code_INSEE'])

In [ ]:
df_election = pd.concat([df_election, df_out], ignore_index=True)
df_election['Code_INSEE'] = df_election['Code_INSEE'].astype(str)
df_election = df_election.sort_values(by=['Code_INSEE']).reset_index(drop=True)
df_election = df_election.drop_duplicates(subset=['Code_INSEE', 'Année']).reset_index(drop=True)
display(df_election)

df_election.to_csv('src/election.csv', index=False, sep=';', encoding='utf-8')